# Module 3 Lab 3: Outliers in the Bridges Data

Krystal-Rose C. Uduma | CS 82A

Input: `bridges.csv` (Canvas course file, saved next to this notebook).

Portfolio: https://github.com/krystalroseuduma/cs82a-portfolio

## 1. Set the plausibility and review rules

I assume a recorded bridge length must be positive. I use the standard 1.5 IQR fences on the observed lengths to flag unusually short or long bridges for review. Crossing a fence tells me to inspect the record and check its context. Blank lengths will be reported separately because they cannot be screened.

In [1]:
import pandas as pd

bridges = pd.read_csv("bridges.csv")
length = bridges["length"]
print("Dataset shape:", bridges.shape)
print("Missing lengths:", length.isna().sum())
print(length.describe().round(2).to_string())

Dataset shape: (108, 13)
Missing lengths: 27
count      81.00
mean     1567.47
std       747.49
min       804.00
25%      1000.00
50%      1300.00
75%      2000.00
max      4558.00


## 2. Show rows that need review

In [2]:
q1, q3 = length.quantile([0.25, 0.75])
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
impossible = length.le(0)
unusual = length.lt(lower) | length.gt(upper)
flags = bridges.loc[impossible | unusual].sort_values("length", ascending=False)

print(f"Q1 = {q1:g}, Q3 = {q3:g}, IQR = {iqr:g}")
print(f"Review fences: {lower:g} to {upper:g}")
print("Nonpositive recorded lengths:", impossible.sum())
print("Rows flagged for review:", len(flags))
print(flags[["Id", "river", "erected", "purpose", "length",
             "material", "span", "type"]].to_string(index=False))

Q1 = 1000, Q3 = 2000, IQR = 1000
Review fences: -500 to 3500
Nonpositive recorded lengths: 0
Rows flagged for review: 3
 Id river  erected purpose  length material span     type
E34     O     1888      RR  4558.0    STEEL LONG SIMPLE-T
E46     A     1897      RR  4000.0    STEEL LONG SIMPLE-T
E91     O     1975 HIGHWAY  3756.0    STEEL LONG     ARCH


## 3. Handle the flagged lengths

E34, E46, and E91 are unusually long relative to this file. They are positive, and the records describe long-span steel bridges, so I kept the measured lengths and flagged them for review. The original Pittsburgh Bridges dataset also documents 4,558 as its maximum recorded length. Capping these values could obscure real differences between bridge purposes. I kept all 108 bridge records and used the 81 observed lengths for the length comparison. Filling the 27 missing lengths with a single estimate could flatten differences between bridge types.

Dataset context: [UCI Pittsburgh Bridges](https://archive.ics.uci.edu/dataset/18/pittsburgh+bridges).

## 4. Prepare comparisons using the available records

In [3]:
reviewed = bridges.assign(length_review=impossible | unusual)
assert len(reviewed) == 108
assert set(flags["Id"]) == {"E34", "E46", "E91"}
assert not impossible.any() and length.isna().sum() == 27

print("Length by purpose, using observed lengths:")
print(reviewed.groupby("purpose")["length"]
      .agg(available="count", median_length="median").to_string())
print("\nLanes by material, using rows with both fields:")
print(pd.crosstab(reviewed["lanes"], reviewed["material"]).to_string())
print("\nLanes by span, using rows with both fields:")
print(pd.crosstab(reviewed["lanes"], reviewed["span"]).to_string())
print("\nMissing fields recorded separately:")
print(reviewed.isna().sum().loc[lambda x: x.gt(0)].to_string())

Length by purpose, using observed lengths:
          available  median_length
purpose                           
AQUEDUCT          2         1046.0
HIGHWAY          57         1240.0
RR               22         1460.5
WALK              0            NaN

Lanes by material, using rows with both fields:
material  IRON  STEEL  WOOD
lanes                      
1.0          1      0     3
2.0          9     40    12
4.0          0     22     1
6.0          0      4     0

Lanes by span, using rows with both fields:
span   LONG  MEDIUM  SHORT
lanes                     
1.0       0       0      2
2.0      14      33      6
4.0      10      12      1
6.0       3       1      0

Missing fields recorded separately:
location     1
length      27
lanes       16
clear-g      2
t-or-d       6
material     2
span        16
rel-l        5
type         2
